# Baselines under the **identical** pooled protocol
## Random Forest · BiLSTM · 1D-CNN · Transformer

 Every baseline is trained and evaluated on
**exactly the same folds, the same per-subject normalisation, and the same windows** as MV-STGNN,
so the per-subject scores are genuinely *paired*

| | |
|---|---|
| Protocol | pooled subject-mixed, repetition-level 5-fold split (test rep ∈ {1,2,4,5,6}, val rep 3) |
| Scope | inherited from the MV-STGNN run — 20 subjects × 5 movement classes, rest excluded |
| Baselines | **RF** (classical features), **BiLSTM**, **1D-CNN**, **Transformer** |
| Artifacts | per-subject scores, per-fold pooled scores, **raw predictions + probabilities**, configs, and one combined long-format table including MV-STGNN |

### The fairness contract (`PLAN.md` §8.2)
A baseline comparison is only worth reporting if the baselines were given a real chance. Every
deep baseline here gets **identically**:

- the same `fold_indices`, `compute_scale`, `PooledSource`, and augmentation suite
- the same optimiser family, LR schedule, warm-up, epoch cap, patience, AMP, grad clipping
- the same class weighting and label smoothing
- the same **SupCon auxiliary loss** — it is a shared toolkit item, not an MV-STGNN privilege
- the same **per-subject FiLM conditioning** — otherwise MV-STGNN's advantage could just be FiLM
- a **matched parameter budget** (~0.15–0.25 M, against MV-STGNN's 0.189 M), reported in the results table

RF gets the same folds, the same normalised signal, and subject identity as a feature, so it is
not handicapped on information either.

### Step map
0. Environment · 1. Config + **compatibility check against the MV-STGNN run** · 2. Shared data
pipeline · 3. Load subjects · 4. Leakage + split-fingerprint gate · 5. Classical features
· 6. Baseline architectures · 7. Shared training loop · 8. Random Forest · 9. Run grid
· 10. Results · 11. Combined artifacts for significance testing

---
## Step 0 — Environment

In [ ]:
import hashlib, json, math, os, random, sys, time, warnings
from dataclasses import dataclass, asdict, replace as dc_replace
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io
import scipy.signal as signal
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             cohen_kappa_score, f1_score)

warnings.filterwarnings("ignore", category=RuntimeWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
BF16 = torch.cuda.is_bf16_supported() if DEVICE.type == "cuda" else False

print("python", sys.version.split()[0], "| numpy", np.__version__,
      "| torch", torch.__version__, "| device", DEVICE, "| bf16", BF16)
try:
    import pywt
    HAVE_PYWT = True
except ImportError:
    HAVE_PYWT = False
print("PyWavelets:", "available (mDWT features enabled)" if HAVE_PYWT
      else "NOT installed -> mDWT features skipped (optional; pip install PyWavelets)")

python 3.11.14 | numpy 2.3.5 | torch 2.10.0+cu128 | device cuda | bf16 True
PyWavelets: NOT installed -> mDWT features skipped (optional; pip install PyWavelets)


---
## Step 1 — Config, and the compatibility check that makes pairing valid

The data/protocol fields below **must** match the MV-STGNN run exactly, or the per-subject scores
are not paired and the Wilcoxon test is meaningless. Rather than trusting that by eye, Step 1
loads the saved MV-STGNN config JSON and **asserts field-by-field equality** on every field that
can change what data a model sees. Model/optimiser fields are allowed to differ.

In [ ]:
PROJECT = Path(r"c:\Users\deskt\Desktop\Nianpro_EMG_Project")
DB2_ROOT = PROJECT / "nina_pro_db_2" / "DB2_Extracted"
RESULTS = PROJECT / "results" / "tables"
RUNS = PROJECT / "results" / "runs" / "baselines"
PREDS = PROJECT / "results" / "preds"
for d in (RESULTS, RUNS, PREDS):
    d.mkdir(parents=True, exist_ok=True)

GNN_TAG = "mvstgnn_pooled_5cls_200ms"          # <- the MV-STGNN run being matched


def _rel(p):
    """Project-relative path for printing, tolerant of paths outside the project."""
    try:
        return str(Path(p).relative_to(PROJECT))
    except ValueError:
        return str(p)


@dataclass
class Cfg:
    # ======== DATA / PROTOCOL — must equal the MV-STGNN run ================
    fs: int = 2000
    n_channels: int = 12
    include_rest: bool = False
    subjects: tuple = tuple(range(1, 21))
    class_subset: tuple = (0, 1, 2, 3, 4)
    bp_low: float = 20.0
    bp_high: float = 450.0
    bp_order: int = 4
    notch_freqs: tuple = (50, 100, 150, 200, 250, 300, 350, 400)
    notch_q: float = 30.0
    trim_ms: int = 50
    win_ms: int = 200
    train_stride_ms: int = 40
    eval_stride_ms: int = 50
    norm_pct: float = 99.9
    ring_channels: tuple = (0, 1, 2, 3, 4, 5, 6, 7)
    floor_frac: float = 0.05
    bad_channels: tuple = ((7, 5), (20, 5))
    val_rep: int = 3
    test_reps: tuple = (1, 2, 4, 5, 6)

    # ======== SHARED TRAINING TOOLKIT (same for every deep model) =========
    label_smoothing: float = 0.05
    w_supcon: float = 0.10
    supcon_tau: float = 0.10
    class_weighted: bool = True
    cap_train_per_class: bool = False
    use_subject_film: bool = True
    subj_emb_dim: int = 16

    lr: float = 1.5e-3
    weight_decay: float = 1e-2
    warmup_epochs: int = 5
    epochs: int = 90                # raised from 70: 3/5 MV-STGNN folds hit the cap
    batch_size: int = 256
    eval_batch: int = 512
    patience: int = 12
    grad_clip: float = 1.0
    amp: bool = True

    aug_noise_snr_db: tuple = (20.0, 35.0)
    aug_amp_scale: tuple = (0.9, 1.1)
    aug_chan_drop_p: float = 0.15
    aug_time_mask_ms: int = 20

    # ======== BASELINE ARCHITECTURES (param-matched to ~0.19 M) ===========
    cnn_channels: tuple = (48, 96, 128)
    cnn_kernel: int = 7
    lstm_patch: int = 10            # 400 samples -> 40 timesteps
    lstm_embed: int = 96
    lstm_hidden: int = 56           # 64 gave 265 k params (1.43x MV-STGNN); 56 -> ~215 k
    lstm_layers: int = 2
    tr_patch: int = 10              # 400 samples -> 40 tokens
    tr_dim: int = 80
    tr_layers: int = 3
    tr_heads: int = 4
    tr_ffn: int = 160
    p_drop: float = 0.2
    p_drop_head: float = 0.3
    head_hidden: int = 256

    # ======== RANDOM FOREST ==============================================
    rf_trees: int = 100
    rf_min_leaf: int = 2
    rf_max_features: str = "sqrt"
    rf_depth_grid: tuple = (None, 20, 30)   # selected on val rep 3 only
    rf_feat_chunk: int = 512               # bounds the AR/FFT working set (~100 MB)
    rf_hist_bins: int = 10
    rf_ar_order: int = 4

    seeds: tuple = (0,)

    @property
    def win(self): return int(round(self.win_ms * self.fs / 1000))
    @property
    def trim(self): return int(round(self.trim_ms * self.fs / 1000))
    @property
    def train_stride(self): return int(round(self.train_stride_ms * self.fs / 1000))
    @property
    def eval_stride(self): return int(round(self.eval_stride_ms * self.fs / 1000))
    @property
    def n_classes(self):
        if self.class_subset is not None:
            return len(self.class_subset)
        return 18 if self.include_rest else 17
    @property
    def label_map(self):
        if self.class_subset is None:
            return None
        return {c: i for i, c in enumerate(self.class_subset)}
    @property
    def n_movement_classes(self):
        return len(self.class_subset) if self.class_subset is not None else 17
    @property
    def n_folds(self): return len(self.test_reps)

    def train_reps(self, test_rep):
        return tuple(r for r in range(1, 7) if r not in (self.val_rep, test_rep))

    def bad_for(self, subject):
        return [c for s, c in self.bad_channels if s == subject]


CFG = Cfg()

# ---- fields that change WHAT DATA a model sees; all must match the GNN run ----
DATA_FIELDS = ("fs", "n_channels", "include_rest", "subjects", "class_subset",
               "bp_low", "bp_high", "bp_order", "notch_freqs", "notch_q",
               "trim_ms", "win_ms", "train_stride_ms", "eval_stride_ms",
               "norm_pct", "ring_channels", "floor_frac", "bad_channels",
               "val_rep", "test_reps")


def latest(pattern):
    hits = sorted(RESULTS.glob(pattern))
    if not hits:
        return None
    named = [p for p in hits if "latest" in p.name]
    return named[0] if named else hits[-1]


gnn_cfg_path = latest(f"{GNN_TAG}_config_*.json")
assert gnn_cfg_path is not None, (
    f"no MV-STGNN config found in {RESULTS}. Run 02_pooled_mvstgnn_5fold.ipynb first.")
GNN_CFG = json.loads(gnn_cfg_path.read_text())
print(f"matching against: {gnn_cfg_path.name}\n")

def canon(v):
    """JSON round-trip so tuples/tuple-of-tuples compare equal to the saved lists."""
    return json.loads(json.dumps(v))


mine = canon(asdict(CFG))
bad = []
for k in DATA_FIELDS:
    a, b = mine.get(k), canon(GNN_CFG.get(k))
    if a != b:
        bad.append(k)
        print(f"  [MISMATCH] {k}: baselines={a}  gnn={b}")
    else:
        print(f"  [ok] {k:18s} {a}")
assert not bad, f"data/protocol fields differ from the MV-STGNN run: {bad}"
print("\n[PASS] every data/protocol field matches -> per-subject scores will be PAIRED")
print(f"\nscope: {len(CFG.subjects)} subjects x {CFG.n_classes} classes  "
      f"chance {1.0/CFG.n_classes:.4f}")
print(f"epochs {CFG.epochs} (raised from the GNN's {GNN_CFG['epochs']}: "
      f"3/5 GNN folds hit the cap; noted in the results table)")

matching against: mvstgnn_pooled_5cls_200ms_config_20260806_233411.json

  [ok] fs                 2000
  [ok] n_channels         12
  [ok] include_rest       False
  [ok] subjects           [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
  [ok] class_subset       [0, 1, 2, 3, 4]
  [ok] bp_low             20.0
  [ok] bp_high            450.0
  [ok] bp_order           4
  [ok] notch_freqs        [50, 100, 150, 200, 250, 300, 350, 400]
  [ok] notch_q            30.0
  [ok] trim_ms            50
  [ok] win_ms             200
  [ok] train_stride_ms    40
  [ok] eval_stride_ms     50
  [ok] norm_pct           99.9
  [ok] ring_channels      [0, 1, 2, 3, 4, 5, 6, 7]
  [ok] floor_frac         0.05
  [ok] bad_channels       [[7, 5], [20, 5]]
  [ok] val_rep            3
  [ok] test_reps          [1, 2, 4, 5, 6]

[PASS] every data/protocol field matches -> per-subject scores will be PAIRED

scope: 20 subjects x 5 classes  chance 0.2000
epochs 90 (raised from the GNN's 70: 

---
## Step 2 — Shared data pipeline

Character-for-character the same logic as notebook 02: filter cascade, segmentation with class
subsetting and label remap, window index, fold construction, per-subject/per-fold normalisation,
GPU-resident pooled buffer, augmentation, and metrics.

Because splits are a **deterministic** function of (config, data) with no RNG, identical config +
identical code ⇒ identical folds. Step 4 verifies this with a fingerprint rather than assuming it.

In [ ]:
def build_filter_cascade(cfg):
    sec = [signal.butter(cfg.bp_order, [cfg.bp_low, cfg.bp_high],
                         btype="band", fs=cfg.fs, output="sos")]
    for f0 in cfg.notch_freqs:
        if f0 < cfg.fs / 2:
            b, a = signal.iirnotch(f0, cfg.notch_q, fs=cfg.fs)
            sec.append(signal.tf2sos(b, a))
    return np.concatenate(sec, axis=0)


SOS = build_filter_cascade(CFG)


def load_raw(subject, cfg=CFG):
    path = DB2_ROOT / f"DB2_s{subject}" / f"S{subject}_E1_A1.mat"
    try:
        m = scipy.io.loadmat(str(path))
        emg = np.asarray(m["emg"], dtype=np.float32)
        rs = np.asarray(m["restimulus"]).ravel().astype(np.int8)
        rr = np.asarray(m["rerepetition"]).ravel().astype(np.int8)
    except NotImplementedError:
        import h5py
        with h5py.File(path, "r") as f:
            emg = np.asarray(f["emg"], dtype=np.float32).T
            rs = np.asarray(f["restimulus"]).ravel().astype(np.int8)
            rr = np.asarray(f["rerepetition"]).ravel().astype(np.int8)
    assert emg.shape[1] == cfg.n_channels and len(rs) == len(rr) == len(emg)
    return emg, rs, rr


def filter_signal(emg, sos):
    return np.ascontiguousarray(
        signal.sosfiltfilt(sos, emg.astype(np.float64), axis=0), dtype=np.float32)


def extract_segments(rs, rr, cfg=CFG):
    key = rs.astype(np.int64) * 100 + rr.astype(np.int64)
    brk = np.flatnonzero(np.diff(key)) + 1
    starts, ends = np.r_[0, brk], np.r_[brk, len(key)]
    raw = [(int(s), int(e), int(rs[s]), int(rr[s])) for s, e in zip(starts, ends)]
    if cfg.include_rest:
        raw = [(s, e, lab, (next((raw[j][3] for j in range(i + 1, len(raw))
                                 if raw[j][2] != 0), 0) if lab == 0 else rep))
               for i, (s, e, lab, rep) in enumerate(raw)]
    lmap = cfg.label_map
    segs, dropped = [], 0
    for s, e, lab, rep in raw:
        if lab == 0 and not cfg.include_rest:
            continue
        if not 1 <= rep <= 6:
            dropped += 1
            continue
        y = 17 if lab == 0 else lab - 1
        if lmap is not None:
            if y not in lmap:
                continue
            y = lmap[y]
        s2, e2 = s + cfg.trim, e - cfg.trim
        if e2 - s2 < cfg.win:
            dropped += 1
            continue
        segs.append((s2, e2, y, rep))
    return segs, dropped


def build_windows(segs, stride, cfg=CFG):
    st_, lb_, rp_, sg_ = [], [], [], []
    for sid, (s, e, y, rep) in enumerate(segs):
        last = e - cfg.win
        if last < s:
            continue
        st = np.arange(s, last + 1, stride, dtype=np.int64)
        st_.append(st)
        lb_.append(np.full(len(st), y, dtype=np.int64))
        rp_.append(np.full(len(st), rep, dtype=np.int64))
        sg_.append(np.full(len(st), sid, dtype=np.int64))
    if not st_:
        return {k: np.zeros(0, np.int64) for k in ("start", "label", "rep", "seg_id")}
    return dict(start=np.concatenate(st_), label=np.concatenate(lb_),
                rep=np.concatenate(rp_), seg_id=np.concatenate(sg_))


def subset(win, mask):
    return {k: v[mask] for k, v in win.items()}


KEYS = ("start", "label", "rep", "seg_id", "subj_idx")


def build_pooled_index(SUBJ, subjects, offsets, stride, reps, cfg=CFG):
    parts = []
    for si, s in enumerate(subjects):
        w = build_windows(SUBJ[s]["segs"], stride, cfg)
        w = subset(w, np.isin(w["rep"], reps))
        w["start"] = w["start"] + offsets[si]
        w["seg_id"] = w["seg_id"] + si * 1000
        w["subj_idx"] = np.full(len(w["start"]), si, dtype=np.int64)
        parts.append(w)
    return {k: np.concatenate([p[k] for p in parts]) for k in KEYS}


def fold_indices(SUBJ, subjects, offsets, fold, cfg=CFG):
    test_rep = cfg.test_reps[fold]
    train_reps = cfg.train_reps(test_rep)
    tr = build_pooled_index(SUBJ, subjects, offsets, cfg.train_stride, train_reps, cfg)
    va = build_pooled_index(SUBJ, subjects, offsets, cfg.eval_stride, (cfg.val_rep,), cfg)
    te = build_pooled_index(SUBJ, subjects, offsets, cfg.eval_stride, (test_rep,), cfg)
    meta = dict(fold=fold, test_rep=test_rep, val_rep=cfg.val_rep,
                train_reps=list(train_reps), n_train=len(tr["start"]),
                n_val=len(va["start"]), n_test=len(te["start"]))
    return tr, va, te, meta


def compute_scale(sig, segs, train_reps, cfg=CFG):
    mask = np.zeros(len(sig), dtype=bool)
    for s, e, _, rep in segs:
        if rep in train_reps:
            mask[s:e] = True
    p = np.percentile(np.abs(sig[mask]), cfg.norm_pct, axis=0).astype(np.float64)
    ref = float(np.median(p[list(cfg.ring_channels)]))
    floor = cfg.floor_frac * ref
    return np.maximum(p, floor).astype(np.float32), int((p < floor).sum())


class PooledSource:
    def __init__(self, n_total, cfg=CFG, device=DEVICE):
        self.cfg, self.device = cfg, device
        self.buf = torch.zeros((n_total, cfg.n_channels), dtype=torch.float16, device=device)
        self.ar = torch.arange(cfg.win, device=device)

    def write(self, offset, sig, scale):
        x = np.clip(sig / scale, -1.0, 1.0)
        self.buf[offset:offset + len(x)] = torch.from_numpy(x).to(self.device, torch.float16)

    def gather(self, starts, train, gen=None):
        idx = starts[:, None] + self.ar[None, :]
        x = self.buf[idx].to(torch.float32).permute(0, 2, 1).contiguous()
        return self._augment(x, gen) if train else x

    def _augment(self, x, gen=None):
        cfg, dev = self.cfg, x.device
        B, Ch, L = x.shape
        r = lambda *s: torch.rand(*s, device=dev, generator=gen)
        lo, hi = cfg.aug_amp_scale
        x = x * (lo + (hi - lo) * r(B, Ch, 1))
        slo, shi = cfg.aug_noise_snr_db
        snr = slo + (shi - slo) * r(B, 1, 1)
        rms = x.pow(2).mean(dim=2, keepdim=True).sqrt()
        x = x + rms * torch.pow(10.0, -snr / 20.0) * torch.randn(
            x.shape, device=dev, generator=gen)
        if cfg.aug_chan_drop_p > 0:
            hit = (r(B, 1) < cfg.aug_chan_drop_p).to(x.dtype)
            which = torch.randint(0, Ch, (B,), device=dev, generator=gen)
            x = x * (1.0 - F.one_hot(which, Ch).to(x.dtype) * hit).unsqueeze(-1)
        span = int(round(cfg.aug_time_mask_ms * cfg.fs / 1000))
        if span > 0:
            t0 = torch.randint(0, max(1, L - span), (B, 1), device=dev, generator=gen)
            ta = torch.arange(L, device=dev)[None, :]
            x = x * (~((ta >= t0) & (ta < t0 + span)))[:, None, :].to(x.dtype)
        return x


class Batcher:
    def __init__(self, win, device=DEVICE):
        self.start = torch.from_numpy(win["start"]).to(device)
        self.label = torch.from_numpy(win["label"]).to(device)
        self.subj = torch.from_numpy(win["subj_idx"]).to(device)
        self.n = len(win["start"])
        self.device = device

    def epoch(self, batch, shuffle, rng=None):
        order = np.arange(self.n)
        if shuffle:
            (rng or np.random).shuffle(order)
        for i in range(0, self.n, batch):
            sel = torch.from_numpy(order[i:i + batch]).to(self.device)
            yield self.start[sel], self.label[sel], self.subj[sel]


def metrics_from(y, p, seg, n_classes):
    out = dict(acc=accuracy_score(y, p), bal_acc=balanced_accuracy_score(y, p),
               macro_f1=f1_score(y, p, average="macro", zero_division=0),
               weighted_f1=f1_score(y, p, average="weighted", zero_division=0),
               kappa=cohen_kappa_score(y, p))
    sy, sp = [], []
    for s in np.unique(seg):
        m = seg == s
        sy.append(y[m][0]); sp.append(np.bincount(p[m], minlength=n_classes).argmax())
    out["vote_acc"] = accuracy_score(sy, sp)
    return out


def per_subject_rows(win, pred, subjects, n_classes):
    rows = []
    for si, s in enumerate(subjects):
        m = win["subj_idx"] == si
        if not m.any():
            continue
        r = metrics_from(win["label"][m], pred[m], win["seg_id"][m], n_classes)
        r["subject"] = s
        r["n_test"] = int(m.sum())
        rows.append(r)
    return rows


def make_class_weights(labels, n_classes, device=DEVICE):
    cnt = np.maximum(np.bincount(labels, minlength=n_classes).astype(np.float64), 1.0)
    return torch.tensor(cnt.sum() / (n_classes * cnt), dtype=torch.float32, device=device)


def cap_train_windows(win, n_classes, seed=0):
    rng = np.random.default_rng(seed)
    keep = []
    cnt = np.bincount(win["label"], minlength=n_classes)
    present = cnt[cnt > 0]
    if not len(present):
        return win
    target = int(present.min())
    for c in range(n_classes):
        i = np.flatnonzero(win["label"] == c)
        if len(i):
            keep.append(rng.choice(i, target, replace=False) if len(i) > target else i)
    return subset(win, np.sort(np.concatenate(keep)))


def supcon_loss(z, y, tau):
    z = F.normalize(z.float(), dim=1)
    sim = z @ z.t() / tau
    n = z.shape[0]
    eye = torch.eye(n, dtype=torch.bool, device=z.device)
    sim = sim.masked_fill(eye, torch.finfo(sim.dtype).min)
    pos = (y[:, None] == y[None, :]) & ~eye
    npos = pos.sum(1)
    valid = npos > 0
    if not valid.any():
        return z.new_zeros(())
    lp = sim - torch.logsumexp(sim, dim=1, keepdim=True)
    contrib = torch.where(pos, lp, torch.zeros_like(lp)).sum(1)
    return -(contrib[valid] / npos[valid]).mean()


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


print("pipeline defined (identical to notebook 02)")

pipeline defined (identical to notebook 02)


---
## Step 3 — Load subjects

In [ ]:
def load_all_subjects(subjects, cfg=CFG, verbose=True):
    SUBJ, t0 = {}, time.time()
    for i, s in enumerate(subjects):
        emg, rs, rr = load_raw(s, cfg)
        bad = cfg.bad_for(s)
        for c in bad:
            emg[:, c] = 0.0
        sig = filter_signal(emg, SOS)
        segs, dropped = extract_segments(rs, rr, cfg)
        if not cfg.include_rest:
            exp = 6 * cfg.n_movement_classes
            assert len(segs) == exp, f"subject {s}: {len(segs)} segments, expected {exp}"
        SUBJ[s] = dict(sig=sig, segs=segs, n=len(sig), bad=bad)
        if verbose and (i % 5 == 0 or i == len(subjects) - 1):
            gb = sum(v["sig"].nbytes for v in SUBJ.values()) / 1e9
            print(f"  [{i+1:>2}/{len(subjects)}] s{s:<3} {len(sig):>9,} samples  "
                  f"{len(segs)} segs  dead {bad or '-'}   RAM {gb:.2f} GB  "
                  f"{time.time()-t0:.0f}s")
    lens = np.array([SUBJ[s]["n"] for s in subjects], dtype=np.int64)
    return SUBJ, np.r_[0, np.cumsum(lens)[:-1]], int(lens.sum())


SUBJECTS = list(CFG.subjects)
print(f"loading {len(SUBJECTS)} subjects ...")
SUBJ, OFFSETS, N_TOTAL = load_all_subjects(SUBJECTS, CFG)
print(f"\nmerged {N_TOTAL:,} samples | RAM {N_TOTAL*12*4/1e9:.2f} GB | "
      f"GPU buffer {N_TOTAL*12*2/1e9:.2f} GB")

loading 20 subjects ...
  [ 1/20] s1   1,808,331 samples  30 segs  dead -   RAM 0.09 GB  2s
  [ 6/20] s6   1,797,578 samples  30 segs  dead -   RAM 0.52 GB  13s
  [11/20] s11  1,803,896 samples  30 segs  dead -   RAM 0.95 GB  23s
  [16/20] s16  1,801,903 samples  30 segs  dead -   RAM 1.38 GB  33s
  [20/20] s20  1,801,389 samples  30 segs  dead [5]   RAM 1.73 GB  41s

merged 36,027,660 samples | RAM 1.73 GB | GPU buffer 0.86 GB


---
## Step 4 — Leakage gate + split fingerprint

Two jobs. First the same leakage checks as notebook 02. Second — and specific to this
notebook — a **split fingerprint**: an MD5 over every fold's sorted `(start, label, rep,
subj_idx)` arrays. It is written into the artifacts so `significance` can prove the baselines
and MV-STGNN were scored on identical windows instead of taking it on trust.

In [ ]:
def leakage_tests(SUBJ, subjects, offsets, n_total, cfg=CFG, verbose=True):
    res = []

    def chk(name, ok, detail=""):
        res.append((name, bool(ok)))
        if verbose:
            print(f"  [{'PASS' if ok else 'FAIL'}] {name}" + (f" — {detail}" if detail else ""))

    lens = np.array([SUBJ[s]["n"] for s in subjects])
    ends = offsets + lens
    chk("subject blocks disjoint & contiguous",
        bool(np.all(offsets[1:] == ends[:-1])) and int(ends[-1]) == n_total)

    for f in range(cfg.n_folds):
        tr, va, te, m = fold_indices(SUBJ, subjects, offsets, f, cfg)
        tag = f"fold{f}(rep{m['test_rep']})"
        ok = all(np.all(w["start"] >= offsets[w["subj_idx"]]) and
                 np.all(w["start"] + cfg.win <= offsets[w["subj_idx"]] + lens[w["subj_idx"]])
                 for w in (tr, va, te))
        chk(f"{tag} windows inside own subject block", ok)
        chk(f"{tag} single test rep for all subjects",
            set(te["rep"]) == {m["test_rep"]} and set(va["rep"]) == {cfg.val_rep}
            and set(tr["rep"]) == set(m["train_reps"]))
        chk(f"{tag} all {len(subjects)} subjects in every split",
            all(len(np.unique(w["subj_idx"])) == len(subjects) for w in (tr, va, te)))
        worst = 0
        for si in range(len(subjects)):
            cov = {}
            for nm, w in (("tr", tr), ("va", va), ("te", te)):
                ws = w["start"][w["subj_idx"] == si] - offsets[si]
                c = np.zeros(lens[si], dtype=bool)
                if len(ws):
                    c[(ws[:, None] + np.arange(cfg.win)[None, :]).ravel()] = True
                cov[nm] = c
            worst = max(worst, int((cov["tr"] & cov["va"]).sum()),
                        int((cov["tr"] & cov["te"]).sum()),
                        int((cov["va"] & cov["te"]).sum()))
        chk(f"{tag} zero sample overlap", worst == 0, f"max shared = {worst}")
        g = [set(w["seg_id"]) for w in (tr, va, te)]
        chk(f"{tag} segments disjoint",
            not (g[0] & g[1]) and not (g[0] & g[2]) and not (g[1] & g[2]))
    chk("val rep never a test rep", cfg.val_rep not in cfg.test_reps)
    return res


def split_fingerprint(SUBJ, subjects, offsets, cfg=CFG):
    """MD5 over every fold's window index -> proves identical scoring windows."""
    h = hashlib.md5()
    for f in range(cfg.n_folds):
        for w in fold_indices(SUBJ, subjects, offsets, f, cfg)[:3]:
            for k in ("start", "label", "rep", "subj_idx"):
                h.update(np.ascontiguousarray(w[k]).tobytes())
    return h.hexdigest()[:16]


print("leakage gate ...")
_r = leakage_tests(SUBJ, SUBJECTS, OFFSETS, N_TOTAL, CFG, verbose=True)
_f = [n for n, ok in _r if not ok]
assert not _f, f"LEAKAGE: {_f}"
print(f"\n{len(_r)}/{len(_r)} checks passed — Leakage gate: GREEN")

SPLIT_FP = split_fingerprint(SUBJ, SUBJECTS, OFFSETS, CFG)
print(f"\nsplit fingerprint: {SPLIT_FP}")
print("  -> written into every artifact; 04_significance asserts it matches the GNN run")

leakage gate ...
  [PASS] subject blocks disjoint & contiguous
  [PASS] fold0(rep1) windows inside own subject block
  [PASS] fold0(rep1) single test rep for all subjects
  [PASS] fold0(rep1) all 20 subjects in every split
  [PASS] fold0(rep1) zero sample overlap — max shared = 0
  [PASS] fold0(rep1) segments disjoint
  [PASS] fold1(rep2) windows inside own subject block
  [PASS] fold1(rep2) single test rep for all subjects
  [PASS] fold1(rep2) all 20 subjects in every split
  [PASS] fold1(rep2) zero sample overlap — max shared = 0
  [PASS] fold1(rep2) segments disjoint
  [PASS] fold2(rep4) windows inside own subject block
  [PASS] fold2(rep4) single test rep for all subjects
  [PASS] fold2(rep4) all 20 subjects in every split
  [PASS] fold2(rep4) zero sample overlap — max shared = 0
  [PASS] fold2(rep4) segments disjoint
  [PASS] fold3(rep5) windows inside own subject block
  [PASS] fold3(rep5) single test rep for all subjects
  [PASS] fold3(rep5) all 20 subjects in every split
  [PAS

---
## Step 5 — Classical features for Random Forest
Overview: 480 dimensions (~40 features × 12 channels) extracted from the same normalized signals for fair comparison.

Feature Set: Time-domain (MAV, RMS, WL, ZC, SSC, WAMP, VAR, IEMG, skew, kurtosis), Hjorth parameters, 4 AR & 4 cepstral coefficients, 10-bin histogram, and frequency-domain metrics.

Augmentations: Subject index appended (matching deep model FiLM); optional mDWT (db7) included if PyWavelets is installed.

In [ ]:
def _ar_coeffs(x, order=4):
    """x (N,L) -> (N,order). Levinson-Durbin on FFT autocorrelation, vectorised over N."""
    N, L = x.shape
    nfft = 1 << int(math.ceil(math.log2(max(2 * L, 2))))
    X = np.fft.rfft(x, n=nfft, axis=1)
    r = np.fft.irfft(np.abs(X) ** 2, n=nfft, axis=1)[:, :order + 1]
    r = r / (r[:, :1] + 1e-12)
    a = np.zeros((N, order + 1), dtype=np.float64)
    a[:, 0] = 1.0
    e = np.ones(N, dtype=np.float64)
    for m in range(1, order + 1):
        acc = r[:, m].copy()
        for i in range(1, m):
            acc += a[:, i] * r[:, m - i]
        k = -acc / (e + 1e-12)
        an = a.copy()
        for i in range(1, m):
            an[:, i] = a[:, i] + k * a[:, m - i]
        an[:, m] = k
        a = an
        e = e * np.maximum(1 - k ** 2, 1e-12)
    return a[:, 1:]


def _cepstral(ar):
    n = ar.shape[1]
    c = np.zeros_like(ar)
    for i in range(n):
        ci = -ar[:, i].copy()
        for k in range(i):
            ci = ci - (1.0 - (k + 1) / (i + 1)) * ar[:, k] * c[:, i - k - 1]
        c[:, i] = ci
    return c


def window_features(x, cfg=CFG):
    """x (B, C, L) float32 normalised -> (B, C*F) float32."""
    B, C, L = x.shape
    f = x.reshape(B * C, L).astype(np.float64)
    out = []

    ax = np.abs(f)
    d1 = np.diff(f, axis=1)
    d2 = np.diff(d1, axis=1)
    mav = ax.mean(1)
    rms = np.sqrt((f ** 2).mean(1))
    var = f.var(1)
    thr = 0.01 * (rms + 1e-12)

    out += [mav, rms, np.abs(d1).sum(1), var, ax.sum(1)]                    # MAV RMS WL VAR IEMG
    out.append(((f[:, :-1] * f[:, 1:] < 0) & (np.abs(d1) > thr[:, None])).sum(1))   # ZC
    out.append(((d1[:, :-1] * d1[:, 1:] < 0) &
                ((np.abs(d1[:, :-1]) > thr[:, None]) |
                 (np.abs(d1[:, 1:]) > thr[:, None]))).sum(1))                       # SSC
    out.append((np.abs(d1) > (0.05 * rms[:, None] + 1e-12)).sum(1))                 # WAMP
    mu = f.mean(1, keepdims=True)
    sd = f.std(1) + 1e-12
    out.append((((f - mu) / sd[:, None]) ** 3).mean(1))                             # skew
    out.append((((f - mu) / sd[:, None]) ** 4).mean(1))                             # kurtosis

    v0 = var + 1e-12
    v1 = d1.var(1) + 1e-12
    v2 = d2.var(1) + 1e-12
    mob = np.sqrt(v1 / v0)
    out += [v0, mob, np.sqrt(v2 / v1) / (mob + 1e-12)]                              # Hjorth

    ar = _ar_coeffs(f, cfg.rf_ar_order)
    out += [ar[:, i] for i in range(ar.shape[1])]
    cep = _cepstral(ar)
    out += [cep[:, i] for i in range(cep.shape[1])]

    edges = np.linspace(-1.0, 1.0, cfg.rf_hist_bins + 1)
    for i in range(cfg.rf_hist_bins):
        lo, hi = edges[i], edges[i + 1]
        out.append(((f >= lo) & (f < hi)).sum(1) / L)                                # HIST

    P = np.abs(np.fft.rfft(f, axis=1)) ** 2
    fr = np.fft.rfftfreq(L, 1.0 / cfg.fs)
    tot = P.sum(1) + 1e-12
    cum = np.cumsum(P, axis=1)
    mdf_i = (cum >= (tot / 2)[:, None]).argmax(1)
    out += [(P * fr).sum(1) / tot,                                                   # MNF
            fr[mdf_i],                                                              # MDF
            fr[P.argmax(1)],                                                        # PKF
            P.mean(1), tot,                                                          # MNP TTP
            (P * fr).sum(1) / tot,                                                   # SM1/TTP
            (P * fr ** 2).sum(1) / tot,                                              # SM2/TTP
            (P * fr ** 3).sum(1) / tot]                                              # SM3/TTP
    lo_b = (fr >= 20) & (fr < 100)
    hi_b = (fr >= 100) & (fr <= 450)
    out.append(P[:, lo_b].sum(1) / (P[:, hi_b].sum(1) + 1e-12))                      # FR

    if HAVE_PYWT:
        import pywt
        co = pywt.wavedec(f, "db7", level=3, axis=1)
        for c_ in co:
            out.append(np.abs(c_).mean(1))
            out.append(np.sqrt((c_ ** 2).mean(1)))

    Fm = np.stack(out, axis=1)                              # (B*C, F)
    Fm = np.nan_to_num(Fm, nan=0.0, posinf=0.0, neginf=0.0)
    return Fm.reshape(B, C * Fm.shape[1]).astype(np.float32)


def extract_features(src, win, cfg=CFG, desc=""):
    """Gather windows from the SAME normalised GPU buffer, then featurise on CPU."""
    n = len(win["start"])
    chunks = []
    t0 = time.time()
    st = torch.from_numpy(win["start"]).to(DEVICE)
    for i in range(0, n, cfg.rf_feat_chunk):
        sl = st[i:i + cfg.rf_feat_chunk]
        with torch.no_grad():
            x = src.gather(sl, train=False).cpu().numpy()
        chunks.append(window_features(x, cfg))
    Fm = np.concatenate(chunks, axis=0)
    subj = win["subj_idx"].astype(np.float32)[:, None]
    Fm = np.concatenate([Fm, subj], axis=1)          # subject identity, same info as FiLM
    if desc:
        print(f"    features {desc}: {Fm.shape} in {time.time()-t0:.0f}s")
    return Fm


print(f"feature bank ready (mDWT {'ON' if HAVE_PYWT else 'off'})")

feature bank ready (mDWT off)


---
## Step 6 — Baseline architectures

Shared Interface: Unified SubjectFiLM integration and forward(x, subj) interface for a single training loop.

1D-CNN (0.15M params): 3 temporal conv blocks ($12 \rightarrow 48 \rightarrow 96 \rightarrow 128$, $k=7$), GroupNorm, GELU, max-pool, and GAP.

BiLSTM (~ 0.20M params): Pure recurrent baseline (no conv frontend); 40 non-overlapping patches ($120$-d), Linear($96$), 2-layer BiLSTM ($64$), and pooling.

Transformer (~0.18M params): 40 patch tokens, Linear($80$) + learned positions, 3 pre-LN encoder layers (4 heads, FFN $160$), and CLS/mean pooling.

In [ ]:
class SubjectFiLM(nn.Module):
    """Identical to MV-STGNN's: zero-init projection -> identity at start."""

    def __init__(self, n_subjects, emb_dim, d):
        super().__init__()
        self.emb = nn.Embedding(n_subjects, emb_dim)
        self.to_gb = nn.Linear(emb_dim, 2 * d)
        nn.init.normal_(self.emb.weight, std=0.02)
        nn.init.zeros_(self.to_gb.weight); nn.init.zeros_(self.to_gb.bias)

    def forward(self, h, subj):                 # h (B,...,d) with d last
        g, b = self.to_gb(self.emb(subj)).chunk(2, dim=-1)
        shape = [h.shape[0]] + [1] * (h.dim() - 2) + [h.shape[-1]]
        return h * (1.0 + g.view(shape)) + b.view(shape)


def gn(c, groups=8):
    return nn.GroupNorm(math.gcd(groups, c) or 1, c)


class CNN1D(nn.Module):
    def __init__(self, cfg, n_subjects):
        super().__init__()
        chs = (cfg.n_channels,) + tuple(cfg.cnn_channels)
        blocks = []
        for i in range(len(cfg.cnn_channels)):
            blocks += [nn.Conv1d(chs[i], chs[i + 1], cfg.cnn_kernel,
                                 padding=cfg.cnn_kernel // 2),
                       gn(chs[i + 1]), nn.GELU(), nn.MaxPool1d(4), nn.Dropout(cfg.p_drop)]
        self.body = nn.Sequential(*blocks)
        d = cfg.cnn_channels[-1]
        self.film = SubjectFiLM(n_subjects, cfg.subj_emb_dim, d) if cfg.use_subject_film else None
        self.head = nn.Sequential(nn.Linear(2 * d, cfg.head_hidden), nn.GELU(),
                                  nn.Dropout(cfg.p_drop_head))
        self.cls = nn.Linear(cfg.head_hidden, cfg.n_classes)

    def forward(self, x, subj=None):
        h = self.body(x)                                  # (B,d,T)
        h = h.transpose(1, 2)                             # (B,T,d)
        if self.film is not None:
            h = self.film(h, subj)
        z = self.head(torch.cat([h.mean(1), h.max(1).values], dim=-1))
        return self.cls(z), z


class BiLSTM(nn.Module):
    def __init__(self, cfg, n_subjects):
        super().__init__()
        self.p = cfg.lstm_patch
        self.embed = nn.Linear(cfg.n_channels * self.p, cfg.lstm_embed)
        self.norm = nn.LayerNorm(cfg.lstm_embed)
        self.rnn = nn.LSTM(cfg.lstm_embed, cfg.lstm_hidden, cfg.lstm_layers,
                           batch_first=True, bidirectional=True,
                           dropout=cfg.p_drop if cfg.lstm_layers > 1 else 0.0)
        d = 2 * cfg.lstm_hidden
        self.film = SubjectFiLM(n_subjects, cfg.subj_emb_dim, d) if cfg.use_subject_film else None
        self.head = nn.Sequential(nn.Linear(2 * d, cfg.head_hidden), nn.GELU(),
                                  nn.Dropout(cfg.p_drop_head))
        self.cls = nn.Linear(cfg.head_hidden, cfg.n_classes)

    def forward(self, x, subj=None):
        B, C, L = x.shape
        t = L // self.p
        h = x[:, :, :t * self.p].reshape(B, C, t, self.p)
        h = h.permute(0, 2, 1, 3).reshape(B, t, C * self.p)     # (B,T,C*p)
        h = self.norm(self.embed(h))
        h, _ = self.rnn(h)                                      # (B,T,2H)
        if self.film is not None:
            h = self.film(h, subj)
        z = self.head(torch.cat([h.mean(1), h[:, -1]], dim=-1))
        return self.cls(z), z


class TransformerEnc(nn.Module):
    def __init__(self, cfg, n_subjects):
        super().__init__()
        self.p = cfg.tr_patch
        d = cfg.tr_dim
        ntok = cfg.win // self.p
        self.embed = nn.Linear(cfg.n_channels * self.p, d)
        self.cls_tok = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.zeros(1, ntok + 1, d))
        nn.init.normal_(self.cls_tok, std=0.02); nn.init.normal_(self.pos, std=0.02)
        layer = nn.TransformerEncoderLayer(d, cfg.tr_heads, cfg.tr_ffn, cfg.p_drop,
                                           activation="gelu", batch_first=True,
                                           norm_first=True)
        self.enc = nn.TransformerEncoder(layer, cfg.tr_layers)
        self.ln = nn.LayerNorm(d)
        self.film = SubjectFiLM(n_subjects, cfg.subj_emb_dim, d) if cfg.use_subject_film else None
        self.head = nn.Sequential(nn.Linear(2 * d, cfg.head_hidden), nn.GELU(),
                                  nn.Dropout(cfg.p_drop_head))
        self.cls = nn.Linear(cfg.head_hidden, cfg.n_classes)

    def forward(self, x, subj=None):
        B, C, L = x.shape
        t = L // self.p
        h = x[:, :, :t * self.p].reshape(B, C, t, self.p)
        h = h.permute(0, 2, 1, 3).reshape(B, t, C * self.p)
        h = self.embed(h)
        h = torch.cat([self.cls_tok.expand(B, -1, -1), h], dim=1) + self.pos[:, :t + 1]
        h = self.ln(self.enc(h))
        if self.film is not None:
            h = self.film(h, subj)
        z = self.head(torch.cat([h[:, 0], h[:, 1:].mean(1)], dim=-1))
        return self.cls(z), z


DEEP = {"cnn1d": CNN1D, "bilstm": BiLSTM, "transformer": TransformerEnc}


def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)


print(f"{'model':<14}{'params':>10}   target 0.15-0.25 M vs MV-STGNN 188,871")
for k, klass in DEEP.items():
    m = klass(CFG, len(SUBJECTS))
    print(f"{k:<14}{n_params(m):>10,}")
    xb = torch.randn(4, CFG.n_channels, CFG.win)
    sb = torch.zeros(4, dtype=torch.long)
    lg, z = m(xb, sb)
    assert lg.shape == (4, CFG.n_classes), lg.shape
    del m
print("\n[PASS] all three forward correctly and are param-matched")

model             params   target 0.15-0.25 M vs MV-STGNN 188,871
cnn1d            194,869
bilstm           219,973
transformer      214,981

[PASS] all three forward correctly and are param-matched


C:\Users\deskt\AppData\Local\Temp\ipykernel_12764\272667511.py:87: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, cfg.tr_layers)


---
## Step 7 — Shared training loop

One loop for all three deep baselines, and the **same loop MV-STGNN used** minus the graph-only
auxiliary terms (link-prediction / adjacency-L1), which have no analogue here. Everything else —
optimiser, cosine schedule with warm-up, AMP, gradient clipping, class weights, label smoothing,
SupCon, early stopping on val rep 3, best-weight restore, single touch of the test set — is
identical.

Predictions **and probabilities** are returned so the artifacts support metrics we have not
computed yet (per-class curves, calibration, McNemar tests).

In [ ]:
def evaluate(model, src, win, cfg, want_prob=True):
    model.eval()
    b = Batcher(win)
    P = []
    amp_on = cfg.amp and BF16 and DEVICE.type == "cuda"
    with torch.no_grad():
        for st, _, sb in b.epoch(cfg.eval_batch, shuffle=False):
            x = src.gather(st, train=False)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp_on):
                logits, _ = model(x, sb)
            P.append(logits.float().softmax(-1).cpu().numpy())
    prob = np.concatenate(P) if P else np.zeros((0, cfg.n_classes), np.float32)
    return prob.argmax(1), (prob if want_prob else None)


def build_fold_source(SUBJ, subjects, offsets, n_total, train_reps, cfg):
    src = PooledSource(n_total, cfg)
    nfl = 0
    for si, s in enumerate(subjects):
        scale, f = compute_scale(SUBJ[s]["sig"], SUBJ[s]["segs"], train_reps, cfg)
        src.write(offsets[si], SUBJ[s]["sig"], scale)
        nfl += f
    return src, nfl


def train_deep_fold(name, SUBJ, subjects, offsets, n_total, fold, cfg,
                    seed=0, verbose=True):
    set_seed(seed)
    t_setup = time.time()
    test_rep = cfg.test_reps[fold]
    train_reps = cfg.train_reps(test_rep)
    src, nfl = build_fold_source(SUBJ, subjects, offsets, n_total, train_reps, cfg)
    tr, va, te, meta = fold_indices(SUBJ, subjects, offsets, fold, cfg)
    if cfg.cap_train_per_class:
        tr = cap_train_windows(tr, cfg.n_classes, seed)
        meta["n_train"] = len(tr["start"])

    model = DEEP[name](cfg, len(subjects)).to(DEVICE)
    w = make_class_weights(tr["label"], cfg.n_classes) if cfg.class_weighted else None
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    spe = max(1, math.ceil(len(tr["start"]) / cfg.batch_size))
    total, warm = cfg.epochs * spe, cfg.warmup_epochs * spe

    def lr_at(step):
        if step < warm:
            return (step + 1) / max(1, warm)
        prog = (step - warm) / max(1, total - warm)
        return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog)))

    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
    gen = torch.Generator(device=DEVICE); gen.manual_seed(seed)
    rng = np.random.default_rng(seed)
    amp_on = cfg.amp and BF16 and DEVICE.type == "cuda"
    btr = Batcher(tr)
    if verbose:
        print(f"    setup {time.time()-t_setup:.0f}s | params {n_params(model):,} | "
              f"train {meta['n_train']:,} val {meta['n_val']:,} test {meta['n_test']:,} | "
              f"{spe} steps/ep | floored {nfl}")

    best_f1, best_state, best_ep, bad, hist = -1.0, None, -1, 0, []
    t0 = time.time()
    for ep in range(cfg.epochs):
        model.train()
        tot = seen = 0
        for st, yb, sb in btr.epoch(cfg.batch_size, True, rng):
            x = src.gather(st, train=True, gen=gen)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=amp_on):
                logits, z = model(x, sb)
                loss = F.cross_entropy(logits, yb, weight=w,
                                       label_smoothing=cfg.label_smoothing)
                if cfg.w_supcon > 0:
                    loss = loss + cfg.w_supcon * supcon_loss(z, yb, cfg.supcon_tau)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"non-finite loss ({name}, ep {ep})")
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step(); sched.step()
            tot += loss.item() * len(yb); seen += len(yb)

        pv, _ = evaluate(model, src, va, cfg, want_prob=False)
        vf1 = f1_score(va["label"], pv, average="macro", zero_division=0)
        hist.append(dict(epoch=ep, train_loss=tot / max(1, seen), val_macro_f1=vf1))
        if vf1 > best_f1:
            best_f1, best_ep, bad = vf1, ep, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
        if verbose and (ep % 10 == 0 or ep == cfg.epochs - 1):
            print(f"      ep {ep:>2}  loss {tot/max(1,seen):.4f}  val_f1 {vf1:.4f}  "
                  f"best@{best_ep}  {(time.time()-t0)/60:.1f}m")
        if bad >= cfg.patience:
            if verbose:
                print(f"      early stop ep {ep} (best {best_ep})")
            break

    model.load_state_dict(best_state)
    pt, prob = evaluate(model, src, te, cfg, want_prob=True)
    common = dict(model=name, fold=fold, seed=seed, test_rep=test_rep,
                  n_train=meta["n_train"], best_epoch=best_ep, epochs_run=len(hist),
                  val_macro_f1=best_f1, n_params=n_params(model),
                  train_min=(time.time() - t0) / 60.0)
    rows = per_subject_rows(te, pt, subjects, cfg.n_classes)
    for r in rows:
        r.update(common)
    pooled = metrics_from(te["label"], pt, te["seg_id"], cfg.n_classes)
    pooled.update(common)
    preds = dict(y_true=te["label"], y_pred=pt, y_prob=prob,
                 subj_idx=te["subj_idx"], seg_id=te["seg_id"], rep=te["rep"])
    del src, model, best_state, btr
    torch.cuda.empty_cache()
    return rows, pooled, preds, hist

---
## Step 8 — Random Forest

Same folds, same normalised signal, same subject information. RF has no epochs, so the val
repetition is used to pick `max_depth` from a small grid — the analogue of early stopping, and
the same "select on rep 3 only, never on test" rule.

In [ ]:
def train_rf_fold(SUBJ, subjects, offsets, n_total, fold, cfg, seed=0, verbose=True):
    set_seed(seed)
    t0 = time.time()
    test_rep = cfg.test_reps[fold]
    train_reps = cfg.train_reps(test_rep)
    src, nfl = build_fold_source(SUBJ, subjects, offsets, n_total, train_reps, cfg)
    tr, va, te, meta = fold_indices(SUBJ, subjects, offsets, fold, cfg)
    if cfg.cap_train_per_class:
        tr = cap_train_windows(tr, cfg.n_classes, seed)
        meta["n_train"] = len(tr["start"])

    Xtr = extract_features(src, tr, cfg, "train" if verbose else "")
    Xva = extract_features(src, va, cfg, "val" if verbose else "")
    Xte = extract_features(src, te, cfg, "test" if verbose else "")
    del src
    torch.cuda.empty_cache()

    # standardise on TRAIN only
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-8
    Xtr = (Xtr - mu) / sd; Xva = (Xva - mu) / sd; Xte = (Xte - mu) / sd

    best = (-1.0, None, None)
    for depth in cfg.rf_depth_grid:
        rf = RandomForestClassifier(
            n_estimators=cfg.rf_trees, max_depth=depth,
            max_features=cfg.rf_max_features, min_samples_leaf=cfg.rf_min_leaf,
            class_weight="balanced_subsample" if cfg.class_weighted else None,
            n_jobs=-1, random_state=seed)
        rf.fit(Xtr, tr["label"])
        vf1 = f1_score(va["label"], rf.predict(Xva), average="macro", zero_division=0)
        if verbose:
            print(f"      max_depth={str(depth):>4}  val_f1 {vf1:.4f}")
        if vf1 > best[0]:
            best = (vf1, depth, rf)
    vf1, depth, rf = best
    if verbose:
        print(f"      selected max_depth={depth} (val_f1 {vf1:.4f}) on rep {cfg.val_rep} only")

    prob = rf.predict_proba(Xte).astype(np.float32)
    pt = prob.argmax(1)
    common = dict(model="rf", fold=fold, seed=seed, test_rep=test_rep,
                  n_train=meta["n_train"], best_epoch=-1, epochs_run=-1,
                  val_macro_f1=vf1, n_params=int(sum(
                      t.tree_.node_count for t in rf.estimators_)),
                  train_min=(time.time() - t0) / 60.0)
    rows = per_subject_rows(te, pt, subjects, cfg.n_classes)
    for r in rows:
        r.update(common)
    pooled = metrics_from(te["label"], pt, te["seg_id"], cfg.n_classes)
    pooled.update(common)
    preds = dict(y_true=te["label"], y_pred=pt, y_prob=prob,
                 subj_idx=te["subj_idx"], seg_id=te["seg_id"], rep=te["rep"])
    return rows, pooled, preds, []


print("RF fold trainer ready ('n_params' for RF = total tree nodes, not weights)")

RF fold trainer ready ('n_params' for RF = total tree nodes, not weights)


---
## Step 9 — Run the grid



**Measured runtime** at 20 subjects × 5 classes (~47 k train windows, ~185 steps/epoch). These
come from a timing run, not a guess — the deep baselines are far cheaper per step than MV-STGNN
because none of them carries the multi-view graph

In [ ]:
MODELS_TO_RUN = ("rf", "cnn1d", "bilstm", "transformer")
SEEDS = (0,)
VERBOSE = True


def cfg_hash(cfg):
    blob = json.dumps({k: (list(v) if isinstance(v, tuple) else v)
                       for k, v in asdict(cfg).items()}, sort_keys=True)
    return hashlib.md5(blob.encode()).hexdigest()[:8]


CFG_HASH = cfg_hash(CFG)
CKPT = RUNS / f"folds_{CFG_HASH}"
CKPT.mkdir(parents=True, exist_ok=True)
print(f"config hash {CFG_HASH} -> {_rel(CKPT)}")
print(f"split fingerprint {SPLIT_FP}")


def run_grid(models, cfg=CFG, seeds=(0,), resume=True, verbose=True):
    sub_all, pol_all = [], []
    t_start = time.time()
    for name in models:
        print(f"\n{'='*70}\n{name.upper()}\n{'='*70}")
        for seed in seeds:
            for fold in range(cfg.n_folds):
                fs = CKPT / f"{name}_f{fold}_s{seed}_per_subject.csv"
                fp = CKPT / f"{name}_f{fold}_s{seed}_pooled.csv"
                npz = PREDS / f"{name}_{CFG_HASH}_f{fold}_s{seed}.npz"
                if resume and fs.exists() and fp.exists() and npz.exists():
                    sub_all.append(pd.read_csv(fs)); pol_all.append(pd.read_csv(fp))
                    print(f"  [fold {fold}] cached  bal_acc "
                          f"{pd.read_csv(fp)['bal_acc'].iloc[0]:.4f}")
                    continue
                print(f"  [fold {fold}] test rep {cfg.test_reps[fold]}")
                fn = train_rf_fold if name == "rf" else (
                    lambda *a, **k: train_deep_fold(name, *a, **k))
                rows, pooled, preds, hist = fn(
                    SUBJ, SUBJECTS, OFFSETS, N_TOTAL, fold, cfg, seed, verbose)
                dfr, dfp = pd.DataFrame(rows), pd.DataFrame([pooled])
                dfr.to_csv(fs, index=False); dfp.to_csv(fp, index=False)
                np.savez_compressed(npz, split_fingerprint=SPLIT_FP,
                                    cfg_hash=CFG_HASH, model=name, fold=fold,
                                    seed=seed, subjects=np.array(SUBJECTS), **preds)
                sub_all.append(dfr); pol_all.append(dfp)
                print(f"    -> acc {pooled['acc']:.4f}  bal_acc {pooled['bal_acc']:.4f}  "
                      f"macro_f1 {pooled['macro_f1']:.4f}  vote {pooled['vote_acc']:.4f}  "
                      f"({pooled['train_min']:.1f} min, elapsed "
                      f"{(time.time()-t_start)/60:.0f} min)")
    return pd.concat(sub_all, ignore_index=True), pd.concat(pol_all, ignore_index=True)


df_sub, df_pol = run_grid(MODELS_TO_RUN, CFG, SEEDS, resume=True, verbose=VERBOSE)
print(f"\ndone: {len(df_sub)} per-subject rows, {len(df_pol)} pooled rows")

config hash d5984109 -> results\runs\baselines\folds_d5984109
split fingerprint 3fed90994b841e0d

RF
  [fold 0] test rep 1
    features train: (46889, 481) in 40s
    features val: (9276, 481) in 8s
    features test: (9679, 481) in 8s
      max_depth=None  val_f1 0.8479
      max_depth=  20  val_f1 0.8460
      max_depth=  30  val_f1 0.8496
      selected max_depth=30 (val_f1 0.8496) on rep 3 only
    -> acc 0.6914  bal_acc 0.6799  macro_f1 0.6841  vote 0.8400  (1.3 min, elapsed 1 min)
  [fold 1] test rep 2
    features train: (47173, 481) in 40s
    features val: (9276, 481) in 8s
    features test: (9457, 481) in 8s
      max_depth=None  val_f1 0.8337
      max_depth=  20  val_f1 0.8306
      max_depth=  30  val_f1 0.8308
      selected max_depth=None (val_f1 0.8337) on rep 3 only
    -> acc 0.7922  bal_acc 0.7865  macro_f1 0.7911  vote 0.9700  (1.3 min, elapsed 3 min)
  [fold 2] test rep 4
    features train: (47151, 481) in 44s
    features val: (9276, 481) in 9s
    features test

C:\Users\deskt\AppData\Local\Temp\ipykernel_12764\272667511.py:87: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, cfg.tr_layers)


    setup 3s | params 214,981 | train 46,889 val 9,276 test 9,679 | 184 steps/ep | floored 2
      ep  0  loss 2.0884  val_f1 0.4340  best@0  0.1m
      ep 10  loss 1.1819  val_f1 0.7203  best@10  0.6m
      ep 20  loss 0.9488  val_f1 0.7460  best@18  1.0m
      ep 30  loss 0.8495  val_f1 0.7613  best@26  1.5m
      early stop ep 38 (best 26)
    -> acc 0.6300  bal_acc 0.6216  macro_f1 0.6244  vote 0.8300  (1.9 min, elapsed 25 min)
  [fold 1] test rep 2


C:\Users\deskt\AppData\Local\Temp\ipykernel_12764\272667511.py:87: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, cfg.tr_layers)


    setup 3s | params 214,981 | train 47,173 val 9,276 test 9,457 | 185 steps/ep | floored 4
      ep  0  loss 2.0954  val_f1 0.4179  best@0  0.1m
      ep 10  loss 1.2339  val_f1 0.6870  best@10  0.5m
      ep 20  loss 0.9885  val_f1 0.7334  best@20  1.0m
      ep 30  loss 0.8703  val_f1 0.7365  best@24  1.5m
      ep 40  loss 0.8052  val_f1 0.7257  best@33  2.0m
      ep 50  loss 0.7630  val_f1 0.7366  best@42  2.5m
      early stop ep 54 (best 42)
    -> acc 0.7058  bal_acc 0.6997  macro_f1 0.7029  vote 0.9500  (2.6 min, elapsed 28 min)
  [fold 2] test rep 4


...

    setup 3s | params 214,981 | train 47,151 val 9,276 test 9,476 | 185 steps/ep | floored 2
      ep  0  loss 2.0913  val_f1 0.4661  best@0  0.1m
      ep 10  loss 1.2322  val_f1 0.6824  best@10  0.5m
      ep 20  loss 0.9930  val_f1 0.7022  best@17  1.0m
      ep 30  loss 0.8751  val_f1 0.7283  best@30  1.5m
      ep 40  loss 0.8067  val_f1 0.7259  best@30  2.0m
      early stop ep 42 (best 30)
    -> acc 0.7142  bal_acc 0.7035  macro_f1 0.7065  vote 0.9700  (2.1 min, elapsed 30 min)
  [fold 3] test rep 5


C:\Users\deskt\AppData\Local\Temp\ipykernel_12764\272667511.py:87: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, cfg.tr_layers)


    setup 3s | params 214,981 | train 47,488 val 9,276 test 9,212 | 186 steps/ep | floored 2
      ep  0  loss 2.0918  val_f1 0.4432  best@0  0.1m
      ep 10  loss 1.2617  val_f1 0.6927  best@10  0.5m
      ep 20  loss 1.0030  val_f1 0.7440  best@20  1.1m
      ep 30  loss 0.8833  val_f1 0.7371  best@22  1.7m
      early stop ep 34 (best 22)
    -> acc 0.7408  bal_acc 0.7370  macro_f1 0.7357  vote 0.9600  (1.9 min, elapsed 32 min)
  [fold 4] test rep 6


C:\Users\deskt\AppData\Local\Temp\ipykernel_12764\272667511.py:87: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(layer, cfg.tr_layers)


    setup 3s | params 214,981 | train 47,207 val 9,276 test 9,426 | 185 steps/ep | floored 5
      ep  0  loss 2.0925  val_f1 0.4640  best@0  0.1m
      ep 10  loss 1.2284  val_f1 0.7151  best@10  0.6m
      ep 20  loss 0.9816  val_f1 0.7476  best@20  1.3m
      ep 30  loss 0.8727  val_f1 0.7446  best@27  1.7m
      ep 40  loss 0.8035  val_f1 0.7535  best@35  2.3m
      ep 50  loss 0.7629  val_f1 0.7665  best@50  2.8m
      ep 60  loss 0.7338  val_f1 0.7617  best@51  3.4m
      early stop ep 63 (best 51)
    -> acc 0.7139  bal_acc 0.7074  macro_f1 0.7058  vote 0.9500  (3.5 min, elapsed 36 min)

done: 400 per-subject rows, 20 pooled rows


---
## Step 10 — Results, with MV-STGNN alongside

Aggregation is the same as `PLAN.md` §16.1: seeds → folds → **one value per subject**, which is
the paired unit the significance test consumes.

In [ ]:
METRICS = ["acc", "bal_acc", "macro_f1", "weighted_f1", "kappa", "vote_acc"]

gnn_sub_path = latest(f"{GNN_TAG}_per_fold_subject_*.csv")
gnn_pol_path = latest(f"{GNN_TAG}_pooled_per_fold_*.csv")
gnn_sub = pd.read_csv(gnn_sub_path)
gnn_pol = pd.read_csv(gnn_pol_path)
gnn_sub["model"] = "mvstgnn"
gnn_pol["model"] = "mvstgnn"
print(f"pulled MV-STGNN from {gnn_sub_path.name}")

ALL_SUB = pd.concat([df_sub, gnn_sub], ignore_index=True)
ALL_POL = pd.concat([df_pol, gnn_pol], ignore_index=True)

# ---- per model: pooled per-fold -------------------------------------------
print("\n" + "=" * 92)
print("POOLED PER-FOLD, BY MODEL")
print("=" * 92)
for m in ALL_POL["model"].unique():
    d = ALL_POL[ALL_POL.model == m].sort_values("fold")
    print(f"\n{m}   params {int(d['n_params'].iloc[0]):,}   "
          f"total {d['train_min'].sum():.0f} min")
    print(d[["fold", "test_rep", "best_epoch", "epochs_run", "val_macro_f1"]
            + METRICS].round(4).to_string(index=False))

# ---- per-subject (the paired unit) ---------------------------------------
per_subject = (ALL_SUB.groupby(["model", "subject", "fold"], as_index=False)[METRICS].mean()
               .groupby(["model", "subject"], as_index=False)[METRICS].mean())

order = (per_subject.groupby("model")["bal_acc"].mean()
         .sort_values(ascending=False).index.tolist())
print("\n" + "=" * 92)
print(f"MODEL COMPARISON — mean over subjects (n={per_subject.subject.nunique()}), "
      f"{CFG.n_classes}-class, chance {1.0/CFG.n_classes:.4f}")
print("=" * 92)
print(f"{'model':<14}{'bal_acc':>9}{'sd':>8}{'acc':>9}{'macro_f1':>10}{'kappa':>8}"
      f"{'vote':>8}{'params':>11}{'min/fold':>10}")
for m in order:
    d = per_subject[per_subject.model == m]
    p = ALL_POL[ALL_POL.model == m]
    print(f"{m:<14}{d['bal_acc'].mean():>9.4f}{d['bal_acc'].std(ddof=1):>8.4f}"
          f"{d['acc'].mean():>9.4f}{d['macro_f1'].mean():>10.4f}"
          f"{d['kappa'].mean():>8.4f}{d['vote_acc'].mean():>8.4f}"
          f"{int(p['n_params'].iloc[0]):>11,}{p['train_min'].mean():>10.1f}")

# ---- head-to-head vs MV-STGNN (descriptive only; tests live in 04) -------
print("\n" + "=" * 92)
print("PAIRED DIFFERENCES vs MV-STGNN  (descriptive — formal tests in 04_significance)")
print("=" * 92)
ref = per_subject[per_subject.model == "mvstgnn"].set_index("subject")["bal_acc"]
print(f"{'baseline':<14}{'mean diff':>11}{'sd':>9}{'win':>6}{'tie':>5}{'loss':>6}")
for m in order:
    if m == "mvstgnn":
        continue
    o = per_subject[per_subject.model == m].set_index("subject")["bal_acc"]
    common = ref.index.intersection(o.index)
    d = (ref.loc[common] - o.loc[common]).values
    print(f"{m:<14}{d.mean():>+11.4f}{d.std(ddof=1):>9.4f}"
          f"{int((d>0).sum()):>6}{int((d==0).sum()):>5}{int((d<0).sum()):>6}")
print("\n(positive mean diff = MV-STGNN better; n is small, so read 04_significance "
      "for corrected p-values and effect sizes before concluding anything)")

pulled MV-STGNN from mvstgnn_pooled_5cls_200ms_per_fold_subject_latest.csv

POOLED PER-FOLD, BY MODEL

rf   params 752,648   total 7 min
 fold  test_rep  best_epoch  epochs_run  val_macro_f1    acc  bal_acc  macro_f1  weighted_f1  kappa  vote_acc
    0         1          -1          -1        0.8496 0.6914   0.6799    0.6841       0.6906 0.6107      0.84
    1         2          -1          -1        0.8337 0.7922   0.7865    0.7911       0.7918 0.7377      0.97
    2         4          -1          -1        0.8273 0.8377   0.8327    0.8336       0.8376 0.7960      0.99
    3         5          -1          -1        0.8465 0.8701   0.8682    0.8674       0.8702 0.8369      1.00
    4         6          -1          -1        0.8470 0.8090   0.8067    0.8043       0.8080 0.7602      0.96

cnn1d   params 194,869   total 9 min
 fold  test_rep  best_epoch  epochs_run  val_macro_f1    acc  bal_acc  macro_f1  weighted_f1  kappa  vote_acc
    0         1          32          45        0.8158 0

---
## Step 11 — Artifacts for the significance test


In [ ]:
STAMP = time.strftime("%Y%m%d_%H%M%S")
TAG = f"pooled_{CFG.n_classes}cls_{CFG.win_ms}ms"


def save2(df, base):
    a = RESULTS / f"{TAG}_{base}_{STAMP}.csv"
    b = RESULTS / f"{TAG}_{base}_latest.csv"
    df.to_csv(a, index=False); df.to_csv(b, index=False)
    return a, b


ps = per_subject.copy()
ps["split_fingerprint"] = SPLIT_FP
ps["n_classes"] = CFG.n_classes
ps["protocol"] = "pooled_subject_mixed_repetition_split"

paths = []
paths += list(save2(ps, "long_per_subject"))
paths += list(save2(ALL_SUB, "long_per_fold_subject"))
paths += list(save2(ALL_POL, "long_pooled_per_fold"))

manifest = dict(
    stamp=STAMP, tag=TAG,
    split_fingerprint=SPLIT_FP,
    baseline_cfg_hash=CFG_HASH,
    gnn_config_file=gnn_cfg_path.name,
    models=sorted(per_subject["model"].unique().tolist()),
    subjects=list(CFG.subjects),
    n_classes=CFG.n_classes,
    class_subset=list(CFG.class_subset) if CFG.class_subset else None,
    chance=1.0 / CFG.n_classes,
    n_folds=CFG.n_folds, val_rep=CFG.val_rep, test_reps=list(CFG.test_reps),
    seeds=list(SEEDS),
    aggregation="seeds -> folds -> one value per subject; unit of analysis = subject",
    primary_endpoint="bal_acc",
    paired_unit="subject",
    n_paired=int(per_subject.subject.nunique()),
    planned_tests=dict(
        omnibus="Friedman over models on subject-level bal_acc",
        posthoc="Wilcoxon signed-rank, MV-STGNN vs each baseline, Holm-corrected",
        effect_size="Cliff's delta + Cohen's d_z",
        ci="BCa bootstrap over subjects, 10000 resamples",
        secondary="LMM  bal_acc ~ model + (1|subject) + (1|subject:fold)"),
    caveat=("per-subject scores come from ONE pooled model per fold, so they are not "
            "independent replications; paired differences against a baseline trained "
            "under the identical protocol remain valid"),
    pred_files=sorted(p.name for p in PREDS.glob(f"*_{CFG_HASH}_*.npz")),
)
for nm in (f"{TAG}_manifest_{STAMP}.json", f"{TAG}_manifest_latest.json"):
    (RESULTS / nm).write_text(json.dumps(manifest, indent=2))
    paths.append(RESULTS / nm)

cfgp = RESULTS / f"{TAG}_baselines_config_{STAMP}.json"
cfgp.write_text(json.dumps({k: (list(v) if isinstance(v, tuple) else v)
                            for k, v in asdict(CFG).items()}, indent=2))
paths.append(cfgp)

print("wrote:")
for p in paths:
    print(f"  {_rel(p)}  ({p.stat().st_size/1024:.1f} KB)")
npzs = sorted(PREDS.glob(f"*_{CFG_HASH}_*.npz"))
print(f"\nprediction files: {len(npzs)} "
      f"({sum(p.stat().st_size for p in npzs)/1e6:.1f} MB total)")
for p in npzs[:3]:
    print(f"  {_rel(p)}")
if len(npzs) > 3:
    print(f"  ... and {len(npzs)-3} more")

# ---- verify the paired matrix is complete and rectangular ----------------
piv = ps.pivot_table(index="subject", columns="model", values="bal_acc")
print(f"\npaired matrix: {piv.shape[0]} subjects x {piv.shape[1]} models")
print(f"missing cells: {int(piv.isna().sum().sum())}")
assert piv.isna().sum().sum() == 0, "paired matrix has holes -> Wilcoxon would drop subjects"
print("[PASS] complete rectangular paired matrix -> ready for 04_significance")
display(piv.round(4))

wrote:
  results\tables\pooled_5cls_200ms_long_per_subject_20260807_203301.csv  (16.9 KB)
  results\tables\pooled_5cls_200ms_long_per_subject_latest.csv  (16.9 KB)
  results\tables\pooled_5cls_200ms_long_per_fold_subject_20260807_203301.csv  (86.3 KB)
  results\tables\pooled_5cls_200ms_long_per_fold_subject_latest.csv  (86.3 KB)
  results\tables\pooled_5cls_200ms_long_pooled_per_fold_20260807_203301.csv  (4.3 KB)
  results\tables\pooled_5cls_200ms_long_pooled_per_fold_latest.csv  (4.3 KB)
  results\tables\pooled_5cls_200ms_manifest_20260807_203301.json  (2.1 KB)
  results\tables\pooled_5cls_200ms_manifest_latest.json  (2.1 KB)
  results\tables\pooled_5cls_200ms_baselines_config_20260807_203301.json  (1.9 KB)

prediction files: 20 (3.6 MB total)
  results\preds\bilstm_d5984109_f0_s0.npz
  results\preds\bilstm_d5984109_f1_s0.npz
  results\preds\bilstm_d5984109_f2_s0.npz
  ... and 17 more

paired matrix: 20 subjects x 5 models
missing cells: 0
[PASS] complete rectangular paired matrix -> 

model,bilstm,cnn1d,mvstgnn,rf,transformer
subject,,,,,
1,0.8387,0.8774,0.9013,0.9051,0.8738
2,0.6547,0.7364,0.7418,0.7470,0.6896
3,0.6978,0.7781,0.9016,0.8329,0.7140
4,0.5269,0.6383,0.7130,0.6544,0.5725
5,0.7159,0.8524,0.9247,0.8939,0.7891
6,0.6685,0.8261,0.8861,0.8680,0.7849
7,0.4461,0.5750,0.6677,0.6697,0.5087
8,0.7966,0.8816,0.9397,0.9149,0.8241
9,0.8561,0.9188,0.9486,0.9196,0.8785
